In [160]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
import pickle

### Análisis de sentimientos

Los modelos Naive Bayes son muy útiles cuando queremos analizar sentimientos, clasificar textos en tópicos o recomendaciones, ya que las características de estos desafíos cumplen muy bien con los supuestos teóricos y metodológicos del modelo.

En este proyecto practicarás con un conjunto de datos para crear un clasificador de reseñas de la tienda de Google Play.

In [161]:
df = pd.read_csv("https://raw.githubusercontent.com/4GeeksAcademy/naive-bayes-project-tutorial/main/playstore_reviews.csv")

df.head()

,package_name,review,polarity
0,com.facebook.katana,privacy at least put some option appear offli...,0
1,com.facebook.katana,"messenger issues ever since the last update, ...",0
2,com.facebook.katana,profile any time my wife or anybody has more ...,0
3,com.facebook.katana,the new features suck for those of us who don...,0
4,com.facebook.katana,forced reload on uploading pic on replying co...,0


In [162]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   package_name  891 non-null    object
 1   review        891 non-null    object
 2   polarity      891 non-null    int64 
dtypes: int64(1), object(2)
memory usage: 21.0+ KB


In [163]:
df.isna().sum()

package_name    0
review          0
polarity        0
dtype: int64

In [164]:
df_name = df.drop("package_name", axis=1)
df_name["review"] = df["review"].str.strip().str.lower()

### Conclusión: 

Procedo a eliminar esta variable, ya que en el enunciado nos lo dice, no es necesaria para mi análisis.

In [165]:
df_name

,review,polarity
0,privacy at least put some option appear offlin...,0
1,"messenger issues ever since the last update, i...",0
2,profile any time my wife or anybody has more t...,0
3,the new features suck for those of us who don'...,0
4,forced reload on uploading pic on replying com...,0
...,...,...
886,loved it i loooooooooooooovvved it because it ...,1
887,all time legendary game the birthday party lev...,1
888,ads are way to heavy listen to the bad reviews...,0
889,fun works perfectly well. ads aren't as annoyi...,1


### Split

In [166]:
X = df_name["review"]
y = df_name["polarity"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, stratify=y, random_state = 42)

In [167]:
vec_model = CountVectorizer(stop_words = "english")
X_train_mod = vec_model.fit_transform(X_train).toarray()
X_test_mod = vec_model.transform(X_test).toarray()

X_train

485    best app i can't believe that it is free! they...
622    good good for slow connection this uc minilite...
854    #1 great game. challenging to the point where ...
536    very reliable syncing but......the web ui look...
728    didn't work couldn't use it with my note 4, it...
                             ...                        
76     i hated it i am able to log in successfully bu...
678    i found this app very fruitful.. and m using i...
80     bugs with contact syncing very frustrated. aft...
414    we have wifi with full internet connection. an...
785    keeps crashing i really do love the browser ov...
Name: review, Length: 712, dtype: object

Creo vectores que me permiten codificar las palabras relevantes, al mismo tiempo filtro para quedarme con esas mismas palabras relevantes que necesito para comenzar el entrenamiento del modelo. 

Ahora comienzo a interactuar con los diferentes modelos disponibles para encontrar el mejor resultado posible. 

>- MultinomialNB
>- BernoulliNB
>- GaussianNB

In [168]:
model_mnb = MultinomialNB()
model_mnb.fit(X_train_mod, y_train)


,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [169]:
y_pred_mnb = model_mnb.predict(X_test_mod)
metric_mnb = accuracy_score(y_test, y_pred_mnb)


In [170]:
metric_mnb

0.8547486033519553

In [171]:

parametros = {"alpha": (0.01, 0.1, 0.5, 1.0), "fit_prior": [True, False]}

random_search_mnb = RandomizedSearchCV(model_mnb, parametros, n_iter = 8, scoring = "accuracy", cv = 5, random_state = 42)
random_search_mnb.fit(X_train_mod, y_train)


,estimator,MultinomialNB()
,param_distributions,"{'alpha': (0.01, ...), 'fit_prior': [True, False]}"
,n_iter,8
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [172]:
random_search_mnb.best_params_, random_search_mnb.best_score_

({'fit_prior': False, 'alpha': 0.01}, 0.8005712597261894)

Se aplicó RandomizedSearchCV para optimizar los hiperparámetros del modelo Multinomial Naive Bayes. Sin embargo, el modelo base obtuvo un mejor rendimiento en el conjunto de test.

In [173]:
model_bnb = BernoulliNB()
model_bnb.fit(X_train_mod, y_train)

,alpha,1.0
,force_alpha,True
,binarize,0.0
,fit_prior,True
,class_prior,None


In [174]:
y_pred_bnb = model_bnb.predict(X_test_mod)
metric_bnb_base = accuracy_score(y_test, y_pred_bnb)

In [175]:
metric_bnb

0.7821229050279329

In [176]:
parametros = {"alpha": (0.01, 0.1, 0.5, 1.0), "fit_prior": [True, False]}

random_search_bnb = RandomizedSearchCV(model_bnb, parametros, n_iter = 8, scoring = "accuracy", cv = 5, random_state = 42)
random_search_bnb.fit(X_train_mod, y_train)

,estimator,BernoulliNB()
,param_distributions,"{'alpha': (0.01, ...), 'fit_prior': [True, False]}"
,n_iter,8
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [177]:
random_search_bnb.best_params_, random_search_bnb.best_score_

({'fit_prior': True, 'alpha': 0.1}, 0.7907416527134837)

BernoulliNB mejoró ligeramente tras la optimización de hiperparámetros, su rendimiento siguió siendo inferior al de MultinomialNB.

In [178]:
model_gnb = GaussianNB()
model_gnb.fit(X_train_mod, y_train)

,priors,None
,var_smoothing,1e-09


In [179]:
y_pred_gnb = model_gnb.predict(X_test_mod)
metric_gnb = accuracy_score(y_test, y_pred_gnb)

In [180]:
metric_gnb

0.8156424581005587

In [181]:
parametros = {"var_smoothing": [1e-9, 1e-8, 1e-7]}

random_search_gnb = RandomizedSearchCV(model_gnb, parametros, n_iter = 3, scoring = "accuracy", cv = 5, random_state = 42)
random_search_gnb.fit(X_train_mod, y_train)

,estimator,GaussianNB()
,param_distributions,"{'var_smoothing': [1e-09, 1e-08, ...]}"
,n_iter,3
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [182]:
random_search_gnb.best_params_, random_search_gnb.best_score_

({'var_smoothing': 1e-09}, 0.7598443809711415)

Gaussian Naive Bayes obtuvo un rendimiento inferior, especialmente tras la optimización de hiperparámetros, por lo cual se descarta totalmente. 

La conclusion final es que nos quemados con el modelo base MultinomialNB ya que genera un 0.85% por lo caul procedo al guardado del mismo. 

In [183]:
with open('/workspaces/Antonio27M-machine-learning/models/modelado-Multinomialnb.pkl', 'wb') as file:
    pickle.dump(model_mnb, file)

In [184]:
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_mod, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [185]:
y_pred_log = log_reg.predict(X_test_mod)
accu_log =accuracy_score(y_test, y_pred_log)
accu_log

0.8324022346368715

Se exploraron modelos alternativos como Regresión Logística. Aunque este modelo obtuvo buenos resultados, no superó al Naive Bayes multinomial, que resultó más adecuado para el tipo de representación del texto utilizada. Por ello, se mantiene MultinomialNB como modelo final

In [186]:

resultados = pd.DataFrame({
    "Modelo": ["MultinomialNB", "MultinomialNB (RandomSearch)", "BernoulliNB", "BernoulliNB (RandomSearch)", "GaussianNB", "GaussianNB (RandomSearch)", "Logistic Regression"],
    "Accuracy": [metric_mnb, random_search_mnb.best_score_, metric_bnb, random_search.best_score_, metric_gnb, random_search_gnb.best_score_, accu_log]})

resultados

,Modelo,Accuracy
0,MultinomialNB,0.854749
1,MultinomialNB (RandomSearch),0.800571
2,BernoulliNB,0.782123
3,BernoulliNB (RandomSearch),0.759844
4,GaussianNB,0.815642
5,GaussianNB (RandomSearch),0.759844
6,Logistic Regression,0.832402
